In [ ]:
import re
import os
import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

import joblib

# plotting
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from IPython.display import display

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# ============= USER SETTINGS ==============
DATA_PATH = 'data.csv'  # change to your dataset path if needed
TEXT_COL = 'Text'
LABEL_COL = 'label'
TEST_SIZE = 0.2
RANDOM_STATE = 42
OUTPUT_DIR = 'artifacts'
NLTK_DOWNLOAD = True
# ==========================================
print("Imports loaded. DATA_PATH =", DATA_PATH)


In [ ]:
if NLTK_DOWNLOAD:
    nltk.download('punkt')
    nltk.download('stopwords')
    nltk.download('wordnet')
    nltk.download('omw-1.4')

STOPWORDS = set(stopwords.words('english'))
LEMMATIZER = WordNetLemmatizer()

print("NLTK resources ready. Stopwords count:", len(STOPWORDS))


In [ ]:
if not Path(DATA_PATH).exists():
    raise FileNotFoundError(f"Data file not found: {DATA_PATH}\nPlease set DATA_PATH to your CSV file containing '{TEXT_COL}' and '{LABEL_COL}' columns.")

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
print("\nNull counts:")
print(df.isnull().sum())

# verify columns
if TEXT_COL not in df.columns:
    raise ValueError(f"Text column '{TEXT_COL}' not found. Columns: {df.columns.tolist()}")
if LABEL_COL not in df.columns:
    raise ValueError(f"Label column '{LABEL_COL}' not found. Columns: {df.columns.tolist()}")

print("\nLabel distribution:")
display(df[LABEL_COL].value_counts(dropna=False))

print("\nSample rows (first 10):")
display(df[[TEXT_COL, LABEL_COL]].head(10))


In [ ]:
def _clean_single(text, remove_stopwords=True):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)             # remove html
    text = re.sub(r'http\S+|www\S+', ' ', text)      # remove urls
    text = re.sub(r'[^a-z\s]', ' ', text)            # remove digits/punct
    text = re.sub(r'\s+', ' ', text).strip()         # collapse spaces
    tokens = word_tokenize(text)
    cleaned_tokens = []
    for tok in tokens:
        if remove_stopwords and tok in STOPWORDS:
            continue
        lemma = LEMMATIZER.lemmatize(tok)
        cleaned_tokens.append(lemma)
    return ' '.join(cleaned_tokens)

print("Cleaning text... (applied to column)")
df = df.dropna(subset=[TEXT_COL]).copy()
df['clean_text'] = df[TEXT_COL].apply(lambda x: _clean_single(x, remove_stopwords=True))
print("Done. Example original -> cleaned (10 samples):")
display(df[[TEXT_COL, 'clean_text']].sample(10, random_state=RANDOM_STATE).reset_index(drop=True))

In [ ]:
def get_text_stats(series):
    lengths = series.fillna('').apply(len)
    word_counts = series.fillna('').apply(lambda x: len(x.split()))
    stats = {
        'count': int(len(series)),
        'avg_chars': float(lengths.mean()),
        'avg_words': float(word_counts.mean()),
        'min_chars': int(lengths.min()),
        'max_chars': int(lengths.max())
    }
    return stats

print("Text statistics on cleaned text:")
stats = get_text_stats(df['clean_text'])
print(json.dumps(stats, indent=2))

def top_n_words(text_series, n=20):
    all_text = ' '.join(text_series.dropna().astype(str))
    words = all_text.split()
    counts = Counter(words)
    return counts.most_common(n)

top_words = top_n_words(df['clean_text'], n=50)
print("\nTop 20 words (cleaned):")
print(top_words[:20])

try:
    text_for_wc = ' '.join(df['clean_text'].dropna().astype(str))
    wc = WordCloud(width=800, height=400, background_color='white').generate(text_for_wc)
    plt.figure(figsize=(12,6))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title('Wordcloud (cleaned text)')
    plt.show()
except Exception as e:
    print("Wordcloud generation failed:", e)

try:
    words, counts = zip(*top_words[:20])
    plt.figure(figsize=(12,6))
    plt.bar(words, counts)
    plt.xticks(rotation=45, ha='right')
    plt.title('Top 20 words')
    plt.show()
except Exception as e:
    print("Top words plotting failed:", e)


In [ ]:
X = df['clean_text']
y = df[LABEL_COL]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)
print("Train size:", X_train.shape[0], "Test size:", X_test.shape[0])

# create vectorizers and fit on training text
count_v = CountVectorizer(ngram_range=(1,1))
tfidf_v = TfidfVectorizer(ngram_range=(1,1))

count_v.fit(X_train)
tfidf_v.fit(X_train)

X_train_tfidf = tfidf_v.transform(X_train)
X_test_tfidf  = tfidf_v.transform(X_test)

print("TF-IDF feature matrix shape (train):", X_train_tfidf.shape)

# show some top features by idf (lowest idf = most common)
try:
    feature_array = np.array(tfidf_v.get_feature_names_out())
    idf_scores = tfidf_v.idf_
    top_n = np.argsort(idf_scores)[:15]
    print("\nTop 15 TF-IDF words (lowest idf -> most common):")
    for idx in top_n:
        print(feature_array[idx], 'idf=', round(idf_scores[idx],3))
except Exception as e:
    print("Could not list TF-IDF features:", e)


In [ ]:
results = []

# Helper inline evaluation (no function defs in pipeline)
def _evaluate_and_report(name, model, X_test_mat, y_test_ser):
    preds = model.predict(X_test_mat)
    acc = accuracy_score(y_test_ser, preds)
    prec = precision_score(y_test_ser, preds, average='weighted', zero_division=0)
    rec = recall_score(y_test_ser, preds, average='weighted', zero_division=0)
    f1 = f1_score(y_test_ser, preds, average='weighted', zero_division=0)
    print(f'== {name} Results ==')
    print('Accuracy:', round(acc,4))
    print('Precision:', round(prec,4))
    print('Recall:', round(rec,4))
    print('F1-score:', round(f1,4))
    print('\nClassification Report:')
    print(classification_report(y_test_ser, preds, zero_division=0))
    print('Confusion Matrix:')
    print(confusion_matrix(y_test_ser, preds))
    return {'model': name, 'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}

# Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_tfidf, y_train)
results.append(_evaluate_and_report('LogisticRegression', lr, X_test_tfidf, y_test))

# Naive Bayes
nb = MultinomialNB()
nb.fit(X_train_tfidf, y_train)
results.append(_evaluate_and_report('MultinomialNB', nb, X_test_tfidf, y_test))

# SVM (linear)
svm = SVC(kernel='linear', probability=True)
svm.fit(X_train_tfidf, y_train)
results.append(_evaluate_and_report('SVM_linear', svm, X_test_tfidf, y_test))

# summary dataframe
results_df = pd.DataFrame(results).sort_values('f1', ascending=False).reset_index(drop=True)
print("\nModel comparison (sorted by F1):")
display(results_df[['model','accuracy','precision','recall','f1']])


In [ ]:
best_row = results_df.iloc[0]
best_model_name = best_row['model']
model_map = {'LogisticRegression': lr, 'MultinomialNB': nb, 'SVM_linear': svm}
best_model = model_map[best_model_name]
print("Best model chosen:", best_model_name)

print("\nPredictions for sample sentences:")
test_sentences = [
    "I absolutely love this product, it worked perfectly!",
    "Terrible experience — will never buy again.",
    "It was okay, neither good nor bad.",
    "Excellent value for money, highly recommended.",
    "The item arrived late and broken."
]

for s in test_sentences:
    cleaned = _clean_single(s, remove_stopwords=True)
    vect = tfidf_v.transform([cleaned])
    pred = best_model.predict(vect)[0]
    prob = None
    if hasattr(best_model, 'predict_proba'):
        prob = best_model.predict_proba(vect)
    print({
        'raw': s,
        'cleaned': cleaned,
        'prediction': pred,
        'probability': prob
    })


In [ ]:
print("=== Short Summary ===")
print("Total samples:", len(df))
print("Train size:", X_train.shape[0], "Test size:", X_test.shape[0])
print("Top model by F1:", best_model_name)
print("Artifacts saved to folder:", OUTPUT_DIR)
